# 🏠 Dubizzle Egypt Apartment Price Estimator
This notebook is designed to run in **Google Colab**. It trains an XGBoost machine learning model to estimate the price of apartments in Egypt based on the scraped dataset.

### 🚀 Getting Started in Colab
1. In Colab, click the **Folder icon** on the left sidebar.
2. Click the **Upload icon** and select your `dubizzle_cleaned.csv` file.
3. Once uploaded, run the cells below by pressing `Shift + Enter`.


In [ ]:
# Install required libraries (Colab usually has these pre-installed)
!pip install xgboost scikit-learn pandas numpy matplotlib seaborn


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('dark_background')
sns.set_palette("muted")


## 1. Load Data
Load the cleaned dataset generated from the data engineering pipeline.


In [ ]:
# Load the dataset
try:
    df = pd.read_csv('dubizzle_cleaned.csv')
    print(f"Dataset loaded successfully: {df.shape[0]:,} rows and {df.shape[1]} columns.")
except FileNotFoundError:
    print("❌ Error: 'dubizzle_cleaned.csv' not found. Please upload it to the Colab filesystem using the folder icon on the left.")


## 2. Data Preprocessing & Class Casting
Select the relevant model features. We drop sparse text data and focus on categorical, numeric, and boolean attributes.


In [ ]:
# Define features
target = 'price_numeric'

numeric_features = ['area_numeric', 'bedrooms', 'bathrooms']
categorical_features = ['city', 'region', 'finish_type', 'view_type']
boolean_features = ['has_garden', 'has_roof', 'has_elevator', 'has_parking', 'has_pool', 'has_security']

# Filter dataset
features = numeric_features + categorical_features + boolean_features
ml_df = df[features + [target]].copy()

# Drop rows missing the target variable or core area
ml_df.dropna(subset=[target, 'area_numeric'], inplace=True)

# Convert booleans to float (0.0 / 1.0)
for col in boolean_features:
    ml_df[col] = ml_df[col].fillna(False).astype(float)

print(f"Data shape after initial parsing: {ml_df.shape}")


## 3. Handle Extreme Outliers
Machine learning models perform better when extreme outliers are removed. We will softly clip the very top and bottom 1% to prevent the model from skewing toward unrealistic multimillion-dollar mansions.


In [ ]:
# Remove top and bottom 1% outliers in price and area
price_lower = ml_df[target].quantile(0.01)
price_upper = ml_df[target].quantile(0.99)
area_lower = ml_df['area_numeric'].quantile(0.01)
area_upper = ml_df['area_numeric'].quantile(0.99)

filtered_df = ml_df[
    (ml_df[target] >= price_lower) & (ml_df[target] <= price_upper) &
    (ml_df['area_numeric'] >= area_lower) & (ml_df['area_numeric'] <= area_upper)
]

print(f"Removed {len(ml_df) - len(filtered_df):,} outlier records.")
ml_df = filtered_df


## 4. Build Machine Learning Pipeline (XGBoost)
We use scikit-learn's `Pipeline` to handle value imputation and One-Hot Encoding modularly.

**Target Transformation:** Property prices are heavily right-skewed. Predicting the **Logarithm of the Price** (`np.log1p`) stabilizes variance and improves accuracy significantly.


In [ ]:
X = ml_df.drop(columns=[target])
# Target transformation: log(1 + price)
y = np.log1p(ml_df[target]) 

# Train-Test Split (80% Train, 20% Evaluate)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training data size: {X_train.shape[0]:,}")
print(f"Testing data size:  {X_test.shape[0]:,}")

# Numeric Transformer: Impute missing values with median, then scale
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical Transformer: Impute missing with 'Unknown', then One-Hot Encode
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Combine preprocessors
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features + boolean_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# Build complete pipeline with XGBoost Regressor
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', XGBRegressor(
        n_estimators=600,
        learning_rate=0.05,
        max_depth=8,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    ))
])


## 5. Train the Model 🔥
Train the XGBoost pipeline. (Takes ~15-30 seconds on Colab CPU).


In [ ]:
print("Training XGBoost Model... Please wait...")
model_pipeline.fit(X_train, y_train)
print("✅ Training Complete!")


## 6. Evaluate Model Performance
Predict on the unseen 20% test set. We must un-transform the log predictions (`np.expm1`) back to actual EGP to evaluate real-world error metrics.


In [ ]:
# Predict and un-transform
y_pred_log = model_pipeline.predict(X_test)
y_pred = np.expm1(y_pred_log)
y_test_real = np.expm1(y_test)

# Calculate error metrics
mae = mean_absolute_error(y_test_real, y_pred)
rmse = np.sqrt(mean_squared_error(y_test_real, y_pred))
r2 = r2_score(y_test_real, y_pred)
mape = np.mean(np.abs((y_test_real - y_pred) / y_test_real)) * 100

print("📊 Final Performance Metrics:")
print(f"R² Score : {r2:.4f} (Accuracy variance explained)")
print(f"MAE      : EGP {mae:,.0f} (Mean absolute price error)")
print(f"RMSE     : EGP {rmse:,.0f}")
print(f"MAPE     : {mape:.2f}% (Average percentage error)")

# Plotting True vs Predicted Prices
plt.figure(figsize=(10, 6))
plt.scatter(y_test_real, y_pred, alpha=0.3, color='#a78bfa', s=10)
plt.plot([y_test_real.min(), y_test_real.max()], [y_test_real.min(), y_test_real.max()], 'r--', lw=2)
plt.xlabel('Actual Price (EGP)')
plt.ylabel('Predicted Price (EGP)')
plt.title('Actual vs Predicted Apartment Prices')
plt.grid(color='#333', linestyle='--', linewidth=0.5)
plt.tight_layout()
plt.show()


## 7. Feature Importance Analysis
Which factors dictate an apartment's price in Egypt? Let's check what the AI learned.


In [ ]:
# Extract encoded names
cat_encoder = model_pipeline.named_steps['preprocessor'].transformers_[1][1].named_steps['onehot']
cat_features_out = cat_encoder.get_feature_names_out(categorical_features)
all_feature_names = numeric_features + boolean_features + list(cat_features_out)

# Extract XGBoost importances
importances = model_pipeline.named_steps['regressor'].feature_importances_

# Zip and format
importance_df = pd.DataFrame({'Feature': all_feature_names, 'Importance': importances})
importance_df = importance_df.sort_values('Importance', ascending=False).head(15)

plt.figure(figsize=(12, 7))
sns.barplot(x='Importance', y='Feature', data=importance_df, palette='Purples_r')
plt.title('Top 15 Most Influential Features for Price')
plt.xlabel('Relative Importance Score')
plt.ylabel('')
plt.tight_layout()
plt.show()


## 8. Interactive Price Inference
Test the model on a completely new, hypothetical apartment to see the estimated value!


In [ ]:
def estimate_price(area, bedrooms, bathrooms, city, region, finish_type='fully_finished', 
                   has_garden=0, has_roof=0, has_pool=0, has_security=1, view_type='landscape'):
    
    # Create a 1-row DataFrame 
    data = {
        'area_numeric': [area],
        'bedrooms': [bedrooms],
        'bathrooms': [bathrooms],
        'city': [city],
        'region': [region],
        'finish_type': [finish_type],
        'view_type': [view_type],
        'has_garden': [float(has_garden)],
        'has_roof': [float(has_roof)],
        'has_elevator': [1.0], # Default yes
        'has_parking': [1.0],  # Default yes
        'has_pool': [float(has_pool)],
        'has_security': [float(has_security)]
    }
    input_df = pd.DataFrame(data)
    
    # Inference + inverse log transform
    log_pred = model_pipeline.predict(input_df)[0]
    return np.expm1(log_pred)

# ----------------- TRY YOUR OWN APARTMENT BELOW -----------------
est_price = estimate_price(
    area=180, 
    bedrooms=3, 
    bathrooms=2, 
    city='New Cairo', 
    region='Cairo East',
    finish_type='super_lux',
    has_security=1,
    has_pool=0
)

print(f"🏢 AI Estimated Property Value:")
print(f"EGP {est_price:,.0f}")
